# Supplementary Tables S1–S4

Produces **exactly** the four supplementary tables referenced in the main text:

| Output | Description | Main-text reference |
|--------|-------------|--------------------|
| **Table S1** | Classification results, **short prompt** variant (counterpart of main Table 1) | §2.1, §2.1 (Llama recovery) |
| **Table S2** | Proposed vs random selection — full 32 pairwise comparisons, both prompt variants | §2.1.2 (p. 8) |
| **Table S3** | Cross-institutional transfer results, both prompt variants | §2.2 (p. 9) |
| **Table S4** | Full per-class precision, recall, F1 across all configs + regex baseline | §3.5 (p. 27) |

Each table is saved as CSV (archival) and LaTeX (manuscript-ready).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
# ── Paths ────────────────────────────────────────────────────────

root_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(
    (c.resolve() for c in root_candidates
     if (c / 'results').exists() and (c / 'notebooks').exists()),
    Path.cwd().resolve(),
)
print(f'Repo root: {REPO_ROOT}')

CONSOLIDATED_DIR = REPO_ROOT / 'analysis' / 'consolidated_loo_tables'
assert CONSOLIDATED_DIR.exists(), f'Missing {CONSOLIDATED_DIR}'

CROSS_DIR = REPO_ROOT / 'analysis' / 'cross_institutional_tables'

SUPP_DIR = REPO_ROOT / 'supplementary_tables'
SUPP_DIR.mkdir(exist_ok=True)

# ── Load consolidated artifacts ──────────────────────────────────
table_1 = pd.read_csv(CONSOLIDATED_DIR / 'table_1.csv')       # macro F1 + CIs
table_3 = pd.read_csv(CONSOLIDATED_DIR / 'table_3.csv')       # pairwise tests

# Per-class results (try with-committee version first)
for candidate in ['table_2_with_committee.csv', 'table_2.csv']:
    p = CONSOLIDATED_DIR / candidate
    if p.exists():
        per_class_raw = pd.read_csv(p)
        print(f'Per-class source: {candidate} ({len(per_class_raw)} rows)')
        break
else:
    raise FileNotFoundError('No per-class table found in consolidated_loo_tables')

# Best-config lookup
best_configs_df = None
for candidate in ['table_2_best_configs_with_committee.csv', 'table_2_best_configs.csv']:
    p = CONSOLIDATED_DIR / candidate
    if p.exists():
        best_configs_df = pd.read_csv(p)
        print(f'Best-config lookup: {candidate}')
        break

if best_configs_df is None:
    print('[WARN] Best-config lookup not found; S4 will default prompt_variant to long.')

for name, df in [('table_1', table_1), ('table_3', table_3)]:
    print(f'{name}: {len(df)} rows, columns: {list(df.columns)}')


In [ ]:
# ── Shared constants ─────────────────────────────────────────────

MODEL_DISPLAY = {
    'meta-llama/Llama-3.2-3B-Instruct': 'Llama-3.2-3B',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16': 'Ministral-3B',
    'google/gemma-3-4b-it': 'Gemma-3-4B',
    'Qwen/Qwen3-4B-Instruct-2507': 'Qwen3-4B',
    'microsoft/MediPhi-Instruct': 'MediPhi',
    'microsoft/Phi-3.5-mini-instruct': 'Phi-3.5-mini',
    'microsoft/Phi-4-mini-instruct': 'Phi-4-mini',
    'COMMITTEE': 'Committee',
    'regex_baseline': 'Regex baseline',
}

MODEL_ORDER = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'microsoft/Phi-4-mini-instruct',
    'microsoft/MediPhi-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16',
    'google/gemma-3-4b-it',
    'Qwen/Qwen3-4B-Instruct-2507',
    'COMMITTEE',
    'regex_baseline',
]

COHORT_ORDER = ['MIMIC', 'Indian']
PROMPT_ORDER = ['short', 'long']
REGIME_ORDER = ['WithUpdate', 'Random']
PROMPT_REGIME_ORDER = ['zero-shot', 'label-only ICL', 'rationale-augmented ICL']
CLASS_ORDER = ['early', 'late', 'unrelated']
CLASS_DISPLAY = {
    'early': 'Early active pregnancy',
    'late': 'Late active pregnancy',
    'unrelated': 'No active pregnancy',
}


def short_name(model):
    return MODEL_DISPLAY.get(model, model)


def sort_df(df):
    """Sort by cohort, prompt, model using categorical ordering."""
    work = df.copy()
    for col, order in [('cohort', COHORT_ORDER), ('prompt_variant', PROMPT_ORDER),
                        ('regime', REGIME_ORDER), ('prompt_regime', PROMPT_REGIME_ORDER)]:
        if col in work.columns:
            work[col] = pd.Categorical(work[col], categories=order, ordered=True)
    if 'model' in work.columns:
        work['model'] = pd.Categorical(work['model'], categories=MODEL_ORDER, ordered=True)
    sort_cols = [c for c in ['cohort', 'prompt_variant', 'model', 'regime',
                              'prompt_regime', 'comparison', 'class'] if c in work.columns]
    return work.sort_values(sort_cols).reset_index(drop=True)


def format_ci(row, f1_col='macro_f1', lo_col='ci_low', hi_col='ci_high'):
    if pd.isna(row[f1_col]):
        return '---'
    return f"{row[f1_col]:.2f} [{row[lo_col]:.2f}, {row[hi_col]:.2f}]"

---
## Table S1: Short-prompt classification results

Counterpart of main-text Table 1 (which shows long prompt only).  
Same structure: 7 models + committee × 2 cohorts × zero-shot / label-only / rationale × proposed / random.

In [ ]:
# ── Table S1: short-prompt results ────────────────────────────────

s1 = table_1.copy()
s1['model_display'] = s1['model'].map(short_name)

# Filter to SHORT prompt only  (+ zero-shot which has no prompt_variant)
s1 = s1[
    (s1['prompt_variant'] == 'short') |
    s1['prompt_variant'].isna()          # zero-shot & regex have NaN
].copy()

s1['f1_ci'] = s1.apply(format_ci, axis=1)
s1 = sort_df(s1)

# Pivot for display: rows = (cohort, model), cols = (regime, prompt_regime)
pivot_s1 = s1.pivot_table(
    index=['cohort', 'model_display'],
    columns=['regime', 'prompt_regime'],
    values='f1_ci',
    aggfunc='first',
)

print('Table S1: Classification performance — SHORT prompt variant')
display(pivot_s1)

# Export
s1_export = s1[[
    'cohort', 'prompt_variant', 'model_display', 'regime',
    'prompt_regime', 'macro_f1', 'ci_low', 'ci_high', 'n_docs'
]].rename(columns={'model_display': 'model'})

s1_export.to_csv(SUPP_DIR / 'table_s1_short_prompt.csv', index=False)
print(f'\nSaved: table_s1_short_prompt.csv  ({len(s1_export)} rows)')

---
## Table S2: Proposed selection vs balanced random — full 32 comparisons

Main-text §2.1.2 (p. 8) reports summary statistics (21 non-significant, 5 vs 6 split).  
This table provides the full breakdown: 7 models + committee × 2 cohorts × 2 prompt variants = 32 rows.  
Comparison: **rationale-augmented ICL, WithUpdate vs Random** at m=4.

In [ ]:
# ── Table S2: proposed vs random selection ──────────────────────

# Identify the comparison key for "proposed (WithUpdate) rationale vs random rationale"
print('Available comparison types in table_3:')
print(table_3['comparison'].value_counts().to_string())
print()

# Filter to proposed-vs-random comparison only
PROPOSED_VS_RANDOM_KEY = 'rationale_WithUpdate vs rationale_Random'
s2 = table_3[table_3['comparison'] == PROPOSED_VS_RANDOM_KEY].copy()

if s2.empty:
    # Try alternative key names
    candidates = [c for c in table_3['comparison'].unique()
                  if 'random' in c.lower() and 'rationale' in c.lower()]
    print(f'[WARN] Key not found. Candidates: {candidates}')
    if candidates:
        s2 = table_3[table_3['comparison'] == candidates[0]].copy()

assert not s2.empty, 'Could not find proposed-vs-random comparison rows in table_3'

s2['model_display'] = s2['model'].map(short_name)
s2 = sort_df(s2)

# Rename columns for clarity
rename_map = {
    'model_display': 'Model',
    'bootstrap_ci_low': 'ci_low',
    'bootstrap_ci_high': 'ci_high',
    'bootstrap_p': 'p_value',
}
# apply renames that exist
s2 = s2.rename(columns={k: v for k, v in rename_map.items() if k in s2.columns})

# Round
for col in ['delta_f1', 'ci_low', 'ci_high', 'p_value']:
    if col in s2.columns:
        s2[col] = s2[col].round(4)

# Select output columns
out_cols = ['cohort', 'prompt_variant', 'Model', 'delta_f1',
            'ci_low', 'ci_high', 'p_value', 'significant']
s2_export = s2[[c for c in out_cols if c in s2.columns]]

# Summary
n_sig = s2_export['significant'].sum()
n_tot = len(s2_export)
n_pos = ((s2_export['significant']) & (s2_export['delta_f1'] > 0)).sum()
n_neg = ((s2_export['significant']) & (s2_export['delta_f1'] < 0)).sum()
print(f'Table S2: Proposed vs Random — {n_tot} comparisons')
print(f'  Non-significant: {n_tot - n_sig}')
print(f'  Significant (proposed better, Δ>0): {n_pos}')
print(f'  Significant (random better, Δ<0):   {n_neg}')
print()
display(s2_export)

s2_export.to_csv(SUPP_DIR / 'table_s2_proposed_vs_random.csv', index=False)
print(f'\nSaved: table_s2_proposed_vs_random.csv  ({n_tot} rows)')

---
## Table S3: Cross-institutional transfer results

Main-text §2.2 (p. 9) reports transfer in both directions for the long prompt.  
This table provides both prompt variants: 7 models + committee × 2 directions × 2 prompt variants.

**Data source:** `analysis/cross_institutional_tables/transfer_table_1.csv`  
(local = `WithUpdate`, cross-institutional = `Transfer`).  
We pivot to show local F1 vs transfer F1 side by side for the rationale-augmented setting.


In [ ]:
# ── Table S3: cross-institutional transfer ─────────────────────

transfer_candidates = [
    CROSS_DIR / 'transfer_table_1.csv',
    CONSOLIDATED_DIR / 'table_transfer.csv',
]

transfer_source = next((p for p in transfer_candidates if p.exists()), None)
if transfer_source is None:
    checked = '\n'.join(f'  - {p}' for p in transfer_candidates)
    raise FileNotFoundError(f'Could not find transfer results. Checked:\n{checked}')

transfer_table_1 = pd.read_csv(transfer_source)
print(f'Transfer source: {transfer_source}')
print('Regime values in transfer table:')
print(transfer_table_1['regime'].value_counts(dropna=False).to_string())
print()


In [ ]:
# ── Build Table S3 ────────────────────────────────────────────────

t1 = transfer_table_1.copy()
t1['model_display'] = t1['model'].map(short_name)

regime_map = {
    'withupdate': 'WithUpdate',
    'local': 'WithUpdate',
    'ping': 'WithUpdate',
    'pingra': 'WithUpdate',
    'transfer': 'Transfer',
    'crosstransfer': 'Transfer',
    'crossping': 'Transfer',
    'crosspingra': 'Transfer',
}
t1['regime'] = t1['regime'].astype(str).map(
    lambda value: regime_map.get(value.replace('-', '').replace('_', '').lower(), value)
)

# Filter to rationale-augmented ICL only (primary config for transfer analysis)
t1_ra = t1[t1['prompt_regime'] == 'rationale-augmented ICL'].copy()

local = t1_ra[t1_ra['regime'] == 'WithUpdate'].copy()
transfer = t1_ra[t1_ra['regime'] == 'Transfer'].copy()

assert not local.empty, 'No local rows found in the transfer results source'
assert not transfer.empty, 'No transfer rows found in the transfer results source'

local = local.rename(columns={
    'macro_f1': 'local_f1', 'ci_low': 'local_ci_low', 'ci_high': 'local_ci_high'
})
transfer = transfer.rename(columns={
    'macro_f1': 'transfer_f1', 'ci_low': 'transfer_ci_low', 'ci_high': 'transfer_ci_high'
})

merge_keys = ['cohort', 'prompt_variant', 'model', 'model_display']
s3 = local[merge_keys + ['local_f1', 'local_ci_low', 'local_ci_high']].merge(
    transfer[merge_keys + ['transfer_f1', 'transfer_ci_low', 'transfer_ci_high']],
    on=merge_keys,
    how='outer',
    validate='one_to_one',
)
s3['delta'] = (s3['local_f1'] - s3['transfer_f1']).round(4)
s3 = sort_df(s3)

# Format for display
s3['local_ci'] = s3.apply(
    lambda r: format_ci(r, 'local_f1', 'local_ci_low', 'local_ci_high'), axis=1
)
s3['transfer_ci'] = s3.apply(
    lambda r: format_ci(r, 'transfer_f1', 'transfer_ci_low', 'transfer_ci_high'), axis=1
)

s3_display = s3[['cohort', 'prompt_variant', 'model_display',
                 'local_ci', 'transfer_ci', 'delta']].rename(
    columns={'model_display': 'Model', 'local_ci': 'Local F1 [CI]',
             'transfer_ci': 'Transfer F1 [CI]', 'delta': 'Δ (local−transfer)'}
)

print(f'Table S3: Cross-institutional transfer — {len(s3_display)} rows')
display(s3_display)

s3.to_csv(SUPP_DIR / 'table_s3_transfer.csv', index=False)
print(f'\nSaved: table_s3_transfer.csv')


---
## Table S4: Full per-class precision, recall, F1

Per-class P/R/F1 for every consolidated classification condition at primary `m = 2`
(zero-shot, label-only ICL, rationale-augmented ICL; `WithUpdate` / `Random` where applicable),
plus prompt-variant-specific regex baselines.


In [ ]:
# ── Table S4: per-class P/R/F1 (full results) ───────────────────

import sys
from sklearn.metrics import precision_recall_fscore_support

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.consolidated_loo_eval import (
    CONDITION_SPECS,
    EXPECTED_SEEDS,
    MACRO_LABELS,
    PRIMARY_M,
    REGEX_MODEL_NAME,
    discover_variant_specs,
    load_condition_source_frames,
    load_gold_labels,
    merge_with_gold,
    _build_seed_level_committee,
    _collapse_committee_predictions,
    _collapse_model_predictions,
    _filter_primary_condition,
)
PRIMARY_M = 4
full_per_class_rows = []
variant_specs = discover_variant_specs()

print('Rebuilding full per-class results from raw prediction stores...')
for spec in variant_specs:
    print(f'  {spec.cohort} / {spec.prompt_variant}')
    gold_df = load_gold_labels(spec.gold_path)
    source_frames = load_condition_source_frames(spec)
    merged_source_frames = {
        source_key: merge_with_gold(frame, gold_df)
        for source_key, frame in source_frames.items()
    }

    for condition in CONDITION_SPECS:
        merged_primary, is_complete = _filter_primary_condition(
            merged_source_frames.get(condition.source_key, pd.DataFrame()),
            condition,
            primary_m=4,
            expected_seeds=EXPECTED_SEEDS,
        )
        condition_ok = (not merged_primary.empty) if condition.zero_shot else bool(is_complete)
        if not condition_ok:
            available_seeds = (
                sorted(merged_primary['seed'].dropna().astype(int).unique().tolist())
                if 'seed' in merged_primary.columns and not merged_primary.empty else []
            )
            print(f'    [WARN] Skipping {condition.key}: available primary seeds = {available_seeds}')
            continue

        model_predictions = _collapse_model_predictions(merged_primary)
        committee_seed = _build_seed_level_committee(merged_primary)
        committee_predictions = _collapse_committee_predictions(committee_seed)

        prediction_frames = {
            str(model_name): model_predictions[model_predictions['model'] == str(model_name)].copy()
            for model_name in spec.models
        }
        prediction_frames['COMMITTEE'] = committee_predictions.copy()

        for model_name, predictions in prediction_frames.items():
            if predictions is None or predictions.empty:
                continue

            precision, recall, f1, support = precision_recall_fscore_support(
                predictions['label'],
                predictions['pred'],
                labels=MACRO_LABELS,
                average=None,
                zero_division=0,
            )
            for idx, label in enumerate(MACRO_LABELS):
                full_per_class_rows.append(
                    {
                        'cohort': spec.cohort,
                        'prompt_variant': spec.prompt_variant,
                        'model': model_name,
                        'regime': condition.regime,
                        'prompt_regime': condition.prompt_regime,
                        'class': label,
                        'precision': float(precision[idx]),
                        'recall': float(recall[idx]),
                        'f1': float(f1[idx]),
                        'support': int(support[idx]),
                    }
                )

regex_per_class_candidates = {
    ('MIMIC', 'short'): [
        REPO_ROOT / 'results' / 'mimic_streamlined_pipeline_small_models_upd_short' / 'regex_baseline' / 'regex_baseline_per_class.csv',
        REPO_ROOT / 'results' / 'mimic_streamlined_pipeline_small_models' / 'regex_baseline' / 'regex_baseline_per_class.csv',
    ],
    ('MIMIC', 'long'): [
        REPO_ROOT / 'results' / 'mimic_streamlined_pipeline_small_models_upd_long' / 'regex_baseline' / 'regex_baseline_per_class.csv',
        REPO_ROOT / 'results' / 'mimic_streamlined_pipeline_small_models' / 'regex_baseline' / 'regex_baseline_per_class.csv',
    ],
    ('Indian', 'short'): [
        REPO_ROOT / 'results' / 'mimic_streamlined_pipeline_small_models_indic_upd_short' / 'regex_baseline' / 'regex_baseline_per_class.csv',
        REPO_ROOT / 'results' / 'indic_streamlined_pipeline_small_models' / 'regex_baseline' / 'regex_baseline_per_class.csv',
        REPO_ROOT / 'results' / 'INDIA_streamlined_pipeline_small_models' / 'regex_baseline' / 'regex_baseline_per_class.csv',
    ],
    ('Indian', 'long'): [
        REPO_ROOT / 'results' / 'mimic_streamlined_pipeline_small_models_indic_upd_long' / 'regex_baseline' / 'regex_baseline_per_class.csv',
        REPO_ROOT / 'results' / 'indic_streamlined_pipeline_small_models' / 'regex_baseline' / 'regex_baseline_per_class.csv',
        REPO_ROOT / 'results' / 'INDIA_streamlined_pipeline_small_models' / 'regex_baseline' / 'regex_baseline_per_class.csv',
    ],
}

for spec in variant_specs:
    regex_path = next(
        (p for p in regex_per_class_candidates.get((spec.cohort, spec.prompt_variant), []) if p.exists()),
        None,
    )
    if regex_path is None:
        print(f'  [WARN] Missing regex per-class metrics for {spec.cohort} ({spec.prompt_variant})')
        continue

    regex_pc = pd.read_csv(regex_path).rename(columns={'label': 'class'})
    regex_pc['cohort'] = spec.cohort
    regex_pc['prompt_variant'] = spec.prompt_variant
    regex_pc['model'] = REGEX_MODEL_NAME
    regex_pc['regime'] = '---'
    regex_pc['prompt_regime'] = '---'
    full_per_class_rows.extend(
        regex_pc[['cohort', 'prompt_variant', 'model', 'regime', 'prompt_regime',
                  'class', 'precision', 'recall', 'f1', 'support']].to_dict('records')
    )
    print(f'    Regex baseline from {regex_path.relative_to(REPO_ROOT)}')

per_class_df = pd.DataFrame(
    full_per_class_rows,
    columns=['cohort', 'prompt_variant', 'model', 'regime', 'prompt_regime',
             'class', 'precision', 'recall', 'f1', 'support'],
)
assert not per_class_df.empty, 'Failed to rebuild full per-class results for Table S4.'

# ── Format ─────────────────────────────────────────────────────
s4 = per_class_df.copy()
s4['cohort'] = pd.Categorical(s4['cohort'], categories=COHORT_ORDER, ordered=True)
s4['prompt_variant'] = pd.Categorical(s4['prompt_variant'], categories=PROMPT_ORDER, ordered=True)
s4['model'] = pd.Categorical(s4['model'], categories=MODEL_ORDER, ordered=True)
s4['regime'] = pd.Categorical(
    s4['regime'].astype(object).where(s4['regime'].notna(), '---'),
    categories=['---', 'WithUpdate', 'Random'],
    ordered=True,
)
s4['prompt_regime'] = pd.Categorical(
    s4['prompt_regime'].astype(object).where(s4['prompt_regime'].notna(), '---'),
    categories=['zero-shot', 'label-only ICL', 'rationale-augmented ICL', '---'],
    ordered=True,
)
s4['class'] = pd.Categorical(s4['class'], categories=CLASS_ORDER, ordered=True)
s4 = s4.sort_values(['cohort', 'prompt_variant', 'model', 'prompt_regime', 'regime', 'class']).reset_index(drop=True)

s4['Model'] = s4['model'].map(short_name)
s4['Class'] = s4['class'].map(CLASS_DISPLAY)

for col in ['precision', 'recall', 'f1']:
    if col in s4.columns:
        s4[col] = s4[col].round(3)

s4_cols = ['cohort', 'prompt_variant', 'Model', 'regime', 'prompt_regime',
           'Class', 'precision', 'recall', 'f1', 'support']
s4_export = s4[[c for c in s4_cols if c in s4.columns]].copy()

print(f'Table S4: Full per-class P/R/F1 — {len(s4_export)} rows')
display(s4_export)

s4_export.to_csv(SUPP_DIR / 'table_s4_per_class.csv', index=False)
print(f'\nSaved: table_s4_per_class.csv')


---
## Summary & Checklist

In [ ]:
print('='*60)
print('SUPPLEMENTARY TABLES — OUTPUT SUMMARY')
print('='*60)
print()

expected = {
    'table_s1_short_prompt.csv':      'Table S1 — Short-prompt classification (§2.1)',
    'table_s2_proposed_vs_random.csv': 'Table S2 — Proposed vs random, 32 comparisons (§2.1.2)',
    'table_s3_transfer.csv':          'Table S3 — Cross-institutional transfer (§2.2)',
    'table_s4_per_class.csv':         'Table S4 — Per-class P/R/F1 (§3.5)',
}

for fname, desc in expected.items():
    path = SUPP_DIR / fname
    if path.exists():
        size_kb = path.stat().st_size / 1024
        status = f'✓  {size_kb:>6.1f} KB'
    else:
        status = '✗  MISSING'
    print(f'  {status}  {fname:<40} {desc}')

print()
print('Main-text SX placeholders to update:')
print('  p.8  §2.1.2  "Supplementary Table SX" → Table S2')
print('  p.9  §2.2    "Supplementary Table SX" → Table S3')
print('  p.27 §3.5    "Supplementary Table SX" → Table S4')